# Reviewer Note: Benchmark Approved, Exact Simulator Not Approved

## Decision

This module is approved as a **historical benchmark engine**. It is **not approved as an exact ETF simulator**.

The distinction matters:
- The engine is mathematically consistent inside its discrete cash-flow model.
- The inputs are proxy instruments, not a complete representation of real ETF cash flows.
- Therefore, the output is suitable for benchmarking, scenario analysis, and relative comparison, but not for claiming exact realized ETF probabilities.

## What Is Approved

- Rolling historical probability estimates over calendar-aligned monthly windows.
- Fee-adjusted hurdle modeling using a multiplicative gross hurdle.
- Discrete DCA evaluation on the same monthly grid as the sampled series.
- Fail-fast behavior when a required ticker is missing.
- Cached data reuse and incremental update behavior for operational efficiency.

## What Is Not Approved

- Exact ETF cash-flow simulation.
- Currency-complete portfolio replication without a translation layer.
- Dividend-timing exactness for all instruments in the proxy set.
- Withholding-tax exactness.
- Tracking-error exactness.
- Silent substitution of missing tickers or partial proxy portfolios.

## Required Assumptions

The engine must state these assumptions whenever probabilities are shown:

- The results are based on historical proxy data, not guaranteed future outcomes.
- Price-index or adjusted-close proxies may understate or distort true total-return behavior.
- Cross-currency proxies are only approximate unless daily FX translation is explicitly modeled.
- The DCA return distribution assumes the chosen contribution schedule and contribution timing are intentional model inputs.
- The fee adjustment is modeled multiplicatively as a gross hurdle derived from the target net annualized return and blended MER.

## Recommended Labeling

Use this wording in UI, notebooks, and reports:

> Approved as benchmark engine. Not approved as exact simulator.

> Outputs are historical proxy estimates subject to currency, dividend, tax, and tracking-error limitations.

## Implementation Boundary

The correct architecture is:

1. Fetch proxy data through `YahooDAO`.
2. Build the synthetic portfolio series in the mathematical engine.
3. Downsample to monthly observations.
4. Apply the fee-adjusted probability engine.
5. Present the results as benchmark probabilities only.

If any required proxy is unavailable, the engine should fail fast rather than silently proceed with a reduced portfolio.

In [ ]:
"""
Reviewer Note: Benchmark Approved, Exact Simulator Not Approved

This module is approved as a historical benchmark engine. It is not approved as an exact ETF simulator.

The distinction matters:
- The engine is mathematically consistent inside its discrete cash-flow model.
- The inputs are proxy instruments, not a complete representation of real ETF cash flows.
- Therefore, the output is suitable for benchmarking, scenario analysis, and relative comparison, but not for claiming exact realized ETF probabilities.

What Is Approved:
- Rolling historical probability estimates over calendar-aligned monthly windows.
- Fee-adjusted hurdle modeling using a multiplicative gross hurdle.
- Discrete DCA evaluation on the same monthly grid as the sampled series.
- Fail-fast behavior when a required ticker is missing.
- Cached data reuse and incremental update behavior for operational efficiency.

What Is Not Approved:
- Exact ETF cash-flow simulation.
- Currency-complete portfolio replication without a translation layer.
- Dividend-timing exactness for all instruments in the proxy set.
- Withholding-tax exactness.
- Tracking-error exactness.
- Silent substitution of missing tickers or partial proxy portfolios.

Required Assumptions:
- The results are based on historical proxy data, not guaranteed future outcomes.
- Price-index or adjusted-close proxies may understate or distort true total-return behavior.
- Cross-currency proxies are only approximate unless daily FX translation is explicitly modeled.
- The DCA return distribution assumes the chosen contribution schedule and contribution timing are intentional model inputs.
- The fee adjustment is modeled multiplicatively as a gross hurdle derived from the target net annualized return and blended MER.

Implementation Boundary:
1. Fetch proxy data through YahooDAO.
2. Build the synthetic portfolio series in the mathematical engine.
3. Downsample to monthly observations.
4. Apply the fee-adjusted probability engine.
5. Present the results as benchmark probabilities only.

If any required proxy is unavailable, the engine should fail fast rather than silently proceed with a reduced portfolio.
"""

In [ ]:
print("\nStep 4: Visualizing distributions...")
if 'test_lump_sum' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Lump-sum histogram
    ax = axes[0]
    returns_pct = [r * 100 for r in test_lump_sum['returns']]
    ax.hist(returns_pct, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(test_lump_sum['stats']['mean']*100, color='green', linestyle='--', linewidth=2, label='Mean')
    ax.axvline(test_lump_sum['stats']['percentile_5']*100, color='red', linestyle=':', linewidth=2, label='5th %ile')
    ax.axvline(test_lump_sum['stats']['percentile_95']*100, color='orange', linestyle=':', linewidth=2, label='95th %ile')
    ax.set_xlabel('Return (%)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'Lump-Sum Returns ({test_horizon}-year horizon)', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # DCA histogram
    ax = axes[1]
    dca_returns_pct = [r * 100 for r in test_dca['returns']]
    ax.hist(dca_returns_pct, bins=50, alpha=0.7, color='teal', edgecolor='black')
    ax.axvline(test_dca['stats']['mean']*100, color='green', linestyle='--', linewidth=2, label='Mean')
    ax.axvline(test_dca['stats']['percentile_5']*100, color='red', linestyle=':', linewidth=2, label='5th %ile')
    ax.axvline(test_dca['stats']['percentile_95']*100, color='orange', linestyle=':', linewidth=2, label='95th %ile')
    ax.set_xlabel('Return (%)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'DCA Returns ({test_horizon}-year horizon)', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualization complete")
else:
    print("✗ No data to visualize")


In [ ]:
print("Step 3: Computing return scenarios (testing 10-year horizon)...")
test_horizon = 10

try:
    # Compute portfolio price series
    portfolio_series = PortfolioReturnsCalculator.compute_portfolio_price_series(
        prices, optimal_weights
    )
    print(f"✓ Portfolio price series computed ({len(portfolio_series)} days)")

    # Lump-sum returns
    horizon_days = PortfolioReturnsCalculator.trading_days_to_index_length(test_horizon)
    lump_sum_result = PortfolioReturnsCalculator.compute_lump_sum_returns(
        portfolio_series, horizon_days
    )
    print(f"✓ Lump-sum returns: {len(lump_sum_result['returns'])} scenarios")
    print(f"  Mean return: {lump_sum_result['stats']['mean']*100:.2f}%")
    print(f"  Std dev: {lump_sum_result['stats']['std']*100:.2f}%")
    print(f"  5th %ile: {lump_sum_result['stats']['percentile_5']*100:.2f}%")
    print(f"  95th %ile: {lump_sum_result['stats']['percentile_95']*100:.2f}%")

    # DCA returns
    dca_result = PortfolioReturnsCalculator.compute_dca_returns(
        prices, optimal_weights, test_horizon,
        monthly_contribution=MONTHLY_DCA_AMOUNT,
        rebalance_freq_months=REBALANCE_FREQ_MONTHS
    )
    print(f"\n✓ DCA returns: {len(dca_result['returns'])} scenarios")
    print(f"  Mean return: {dca_result['stats']['mean']*100:.2f}%")
    print(f"  Std dev: {dca_result['stats']['std']*100:.2f}%")
    print(f"  5th %ile: {dca_result['stats']['percentile_5']*100:.2f}%")
    print(f"  95th %ile: {dca_result['stats']['percentile_95']*100:.2f}%")

    # Store for visualization
    test_lump_sum = lump_sum_result
    test_dca = dca_result

except Exception as e:
    print(f"✗ Error computing scenarios: {e}")
    import traceback
    traceback.print_exc()


## Section 5: Run a Basic Execution Test

Compute return scenarios for one horizon and visualize the results.

In [ ]:
print("Validating configuration...")
errors = []

if not TICKERS or len(TICKERS) == 0:
    errors.append("Must provide at least one ticker")
if not HORIZONS or len(HORIZONS) == 0:
    errors.append("Must provide at least one horizon")
if MONTHLY_DCA_AMOUNT <= 0:
    errors.append("Monthly DCA must be positive")
if any(h <= 0 for h in HORIZONS):
    errors.append("All horizons must be positive")

if errors:
    print("✗ Validation failed:")
    for err in errors:
        print(f"  - {err}")
else:
    print("✓ Configuration validated")
    print(f"  {len(TICKERS)} tickers, {len(HORIZONS)} horizons")


## Section 4: Add Input Validation

Validate configuration and data before proceeding with calculations.

In [ ]:
print("\nStep 2: Running portfolio optimization...")
try:
    optimizer.fetch_prices = lambda t: prices[t]  # Use fetched or synthetic prices
    opt_result = optimizer.optimize(TICKERS)
    
    print(f"✓ Optimization complete\n")
    print("Optimal Allocation (Max Sharpe Ratio):")
    for ticker, weight in opt_result['optimal']['weights'].items():
        print(f"  {ticker}: {weight * 100:6.2f}%")
    
    print(f"\nOptimal Portfolio Metrics:")
    print(f"  Expected Return: {opt_result['optimal']['expected_return'] * 100:.2f}%")
    print(f"  Volatility:      {opt_result['optimal']['volatility'] * 100:.2f}%")
    print(f"  Sharpe Ratio:    {opt_result['optimal']['sharpe_ratio']:.3f}")
    
    optimal_weights = opt_result['optimal']['weights']
    
except Exception as e:
    print(f"✗ Error optimizing: {e}")
    # Fallback to equal weights
    optimal_weights = {ticker: 1/len(TICKERS) for ticker in TICKERS}
    print(f"  Using equal-weight fallback: {optimal_weights}")


In [ ]:
print("Step 1: Fetching historical prices...")
try:
    optimizer = PortfolioOptimizer(lookback_years=5)
    prices = optimizer.fetch_prices(TICKERS)
    print(f"✓ Fetched {len(prices)} trading days")
    print(f"  Price range: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")
    print(f"\nPrice data head:")
    print(prices.head())
except Exception as e:
    print(f"✗ Error fetching prices: {e}")
    print("  (This may occur if yfinance has network issues; continuing with synthetic data...)")
    # Fallback to synthetic data for demo purposes
    dates = pd.date_range(end=datetime.now(), periods=1260, freq='D')  # ~5 years
    np.random.seed(42)
    prices_dict = {}
    for ticker in TICKERS:
        prices_dict[ticker] = 100 * np.cumprod(1 + np.random.randn(len(dates)) * 0.02)
    prices = pd.DataFrame(prices_dict, index=dates)
    print(f"✓ Using synthetic price data ({len(prices)} days)")


## Section 3: Implement the First Feature — Portfolio Optimization

Fetch historical prices and optimize portfolio weights to maximize Sharpe ratio.

In [ ]:
# Define portfolio ETFs
TICKERS = ['XIU.TO', 'XWD.TO', 'XEU.TO']  # CAD equity, US/Intl equity, EU equity

# Investment horizons (years)
HORIZONS = [2, 5, 10, 15, 25]

# DCA parameters
MONTHLY_DCA_AMOUNT = 1000  # $1000/month
REBALANCE_FREQ_MONTHS = 1  # Monthly rebalancing

# Configuration summary
config = {
    'tickers': TICKERS,
    'horizons_years': HORIZONS,
    'monthly_dca_amount': MONTHLY_DCA_AMOUNT,
    'rebalance_freq_months': REBALANCE_FREQ_MONTHS,
    'lookback_period_years': 5,  # For optimization
}

print("Portfolio Configuration:")
print(f"  Tickers: {config['tickers']}")
print(f"  Horizons: {config['horizons_years']}")
print(f"  Monthly DCA: ${config['monthly_dca_amount']}")
print(f"  Rebalancing: Every {config['rebalance_freq_months']} month(s)")
print(f"  Optimization lookback: {config['lookback_period_years']} years")


## Section 2: Define the Core Data Structures

Set up portfolio composition, horizons, and DCA parameters.

In [ ]:
import sys
import os

# Add the project root to the path so we can import planning modules
sys.path.insert(0, '/Users/Louis-Philippe/Documents/finance_agent')

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Import our custom modules
from planning.optimization import PortfolioOptimizer
from planning.returns import PortfolioReturnsCalculator

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")
print(f"✓ Project path: {sys.path[0]}")


## Section 1: Set Up the Development Environment

Import libraries, configure paths, and set up plotting and data manipulation tools.

# Portfolio Scenario Analysis — Distribution of Returns

This notebook demonstrates the portfolio scenario analysis engine, which computes return distributions for multiple horizons and investment patterns (lump-sum vs. dollar-cost averaging).

**Methodology:**
- Fetch historical adjusted close prices for a portfolio of ETFs (handles dividends implicitly).
- Compute overlapping N-horizon rolling-window returns (e.g., all possible H-year investments).
- Simulate DCA with monthly rebalancing to fixed target weights.
- Visualize return distributions via histograms, KDE, and key percentiles.
- Optimize portfolio weights to maximize Sharpe ratio.